# CareLoop Clinics — No-Show Guard
## Part 1 · Ingest, validate, clean, enrich (Tasks 1.1–1.6)

**Input:** `data/appointments.csv`, `data/clinics.csv` (originals, never edited)
**Outputs:** `rejected_rows.json`, `clinics_enriched.csv`, `appointments_clean.csv`, `load_data.sql`

In Colab, upload the three original CSVs into a folder named `data/` (or upload the whole repo) before running.

In [1]:
import csv
import json
import numpy as np
import pandas as pd
import requests

DATA_DIR = "data/"
RAW_FILE = DATA_DIR + "appointments.csv"
CLINICS_FILE = DATA_DIR + "clinics.csv"

## Task 1.1 — Read the file the foundations way (csv module only)
A **type check**: each row must convert `age`, `fee_inr`, `prior_no_shows` to `int` and `distance_km` to `float`.
Rows that raise `ValueError` are rejected. Note that this does **not** check validity: an age of 250 converts fine.

In [2]:
with open(RAW_FILE, newline="") as f:
    reader = csv.DictReader(f)
    header = reader.fieldnames
    rows = list(reader)

print("Header:", header)
print("Row count:", len(rows))

Header: ['appointment_id', 'patient_id', 'clinic_id', 'booked_on', 'appt_date', 'age', 'gender', 'distance_km', 'fee_inr', 'booking_channel', 'visit_type', 'reminder_sent', 'prior_no_shows', 'status']
Row count: 1020


In [3]:
def parse_row(row):
    """Return a copy of the row with numeric fields converted. Raises ValueError on bad values."""
    parsed = dict(row)
    parsed["age"] = int(row["age"])
    parsed["fee_inr"] = int(row["fee_inr"])
    parsed["prior_no_shows"] = int(row["prior_no_shows"])
    parsed["distance_km"] = float(row["distance_km"])
    return parsed

accepted, rejected = [], []
for row in rows:
    try:
        accepted.append(parse_row(row))
    except ValueError:
        rejected.append(row)

print("Accepted:", len(accepted))
print("Rejected:", len(rejected))
print("  blank age        :", sum(1 for r in rejected if r["age"].strip() == ""))
print("  blank distance_km:", sum(1 for r in rejected if r["distance_km"].strip() == ""))
print("Accepted rows with impossible age (<1 or >100):",
      sum(1 for r in accepted if r["age"] < 1 or r["age"] > 100))

with open("rejected_rows.json", "w") as f:
    json.dump(rejected, f, indent=2)
print("Saved rejected_rows.json")

Accepted: 988
Rejected: 32
  blank age        : 12
  blank distance_km: 20
Accepted rows with impossible age (<1 or >100): 5
Saved rejected_rows.json


## Task 1.2 — Inspect and clean, pass 1 (pandas)
Starts again from the original file; the csv-module lists above are not used from here on.

In [4]:
df = pd.read_csv(RAW_FILE)
print("Shape:", df.shape)
df.info()

Shape: (1020, 14)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1020 entries, 0 to 1019
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   appointment_id   1020 non-null   object 
 1   patient_id       1020 non-null   object 
 2   clinic_id        1020 non-null   object 
 3   booked_on        1020 non-null   object 
 4   appt_date        1019 non-null   object 
 5   age              1008 non-null   float64
 6   gender           1020 non-null   object 
 7   distance_km      1000 non-null   float64
 8   fee_inr          1020 non-null   int64  
 9   booking_channel  1020 non-null   object 
 10  visit_type       1020 non-null   object 
 11  reminder_sent    1020 non-null   object 
 12  prior_no_shows   1020 non-null   int64  
 13  status           1016 non-null   object 
dtypes: float64(2), int64(2), object(10)
memory usage: 111.7+ KB


In [5]:
print("Nulls per column:")
print(df.isnull().sum())
print("\nExact duplicate rows:", df.duplicated().sum())
print("Raw booking_channel spellings:", df["booking_channel"].nunique())
print(df["booking_channel"].value_counts().to_string())

Nulls per column:
appointment_id      0
patient_id          0
clinic_id           0
booked_on           0
appt_date           1
age                12
gender              0
distance_km        20
fee_inr             0
booking_channel     0
visit_type          0
reminder_sent       0
prior_no_shows      0
status              4
dtype: int64

Exact duplicate rows: 20
Raw booking_channel spellings: 16
booking_channel
App         369
Phone       345
Walk-in     236
APP          10
Walk-in      10
app           9
PHONE         7
 walk-in      5
App           5
Walk-In       5
phone         5
 App          5
 phone        4
WALK-IN       3
Phone         1
walk-in       1


In [6]:
print("Start:", len(df))

# (a) drop exact duplicates
df = df.drop_duplicates()
print("(a) after dropping duplicates:", len(df))

# (b) normalise booking_channel labels
df["booking_channel"] = df["booking_channel"].str.strip().str.title()
print("(b) booking_channel values after cleaning:", sorted(df["booking_channel"].unique()))
print("    rows:", len(df))

# (c) convert dates; unreadable appt_date text becomes NaT
before_nat = df["appt_date"].isna().sum()
df["appt_date"] = pd.to_datetime(df["appt_date"], errors="coerce")
df["booked_on"] = pd.to_datetime(df["booked_on"])
print("(c) appt_date missing after conversion:", df["appt_date"].isna().sum(),
      f"(was {before_nat} before conversion)")
print("    rows:", len(df))

# (d) drop rows with no usable appt_date or no status
df = df.dropna(subset=["appt_date", "status"])
print("(d) after dropping missing appt_date / status:", len(df))

Start: 1020
(a) after dropping duplicates: 1000
(b) booking_channel values after cleaning: ['App', 'Phone', 'Walk-In']
    rows: 1000
(c) appt_date missing after conversion: 6 (was 1 before conversion)
    rows: 1000
(d) after dropping missing appt_date / status: 990


## Task 1.3 — Clean, pass 2: validity, gaps, outliers
Order matters: invalid ages → fill gaps → IQR fences (computed on the filled columns) → rounding.

In [7]:
# (a) impossible ages become missing
invalid_age = (df["age"] < 1) | (df["age"] > 100)
print("Invalid ages:", invalid_age.sum(), "->", df.loc[invalid_age, "age"].tolist())
df.loc[invalid_age, "age"] = np.nan
print("Missing ages now:", df["age"].isna().sum())

# (b) fill gaps with the median
age_median = df["age"].median()
dist_median = df["distance_km"].median()
print("Median age:", age_median, "| median distance_km:", dist_median)
df["age"] = df["age"].fillna(age_median).round().astype(int)
df["distance_km"] = df["distance_km"].fillna(dist_median)

Invalid ages: 5 -> [0.0, 150.0, -4.0, 250.0, 199.0]
Missing ages now: 17
Median age: 48.0 | median distance_km: 8.25


In [8]:
# (c) cap outliers at the IQR fences
for col in ["distance_km", "fee_inr"]:
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    iqr = q3 - q1
    low, high = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    n_high = (df[col] > high).sum()
    n_low = (df[col] < low).sum()
    df[col] = np.clip(df[col], low, high)
    print(f"{col}: Q1={q1}, Q3={q3}, IQR={iqr}, fences=[{low}, {high}] "
          f"-> capped {n_high + n_low} values ({n_high} high, {n_low} low)")

# (d) rounding
df["distance_km"] = df["distance_km"].round(1)
df["fee_inr"] = df["fee_inr"].round().astype(int)
print("Largest fee after capping:", df["fee_inr"].max())

distance_km: Q1=4.625, Q3=13.0, IQR=8.375, fences=[-7.9375, 25.5625] -> capped 26 values (26 high, 0 low)
fee_inr: Q1=500.0, Q3=787.5, IQR=287.5, fences=[68.75, 1218.75] -> capped 3 values (3 high, 0 low)
Largest fee after capping: 1219


## Task 1.4 — Features and a NumPy summary

In [9]:
df["lead_days"] = (df["appt_date"] - df["booked_on"]).dt.days
df["no_show"] = (df["status"] == "No-Show").astype(int)

print("lead_days range:", df["lead_days"].min(), "to", df["lead_days"].max())
print(f"No-shows: {df['no_show'].sum()} of {len(df)} ({df['no_show'].mean() * 100:.1f}%)")

lead_days range: 1 to 45
No-shows: 218 of 990 (22.0%)


In [10]:
X = df[["age", "distance_km", "fee_inr", "lead_days"]].to_numpy()

# (a) column means
print("Column means [age, distance_km, fee_inr, lead_days]:", np.round(X.mean(axis=0), 2))

# (b) boolean mask on lead_days
mask = X[:, 3] > 14
print(f"Rows with lead_days > 14: {mask.sum()} ({mask.mean() * 100:.1f}%)")

# (c) z-scores of distance_km (population std, np.std default)
x = X[:, 1]
z = (x - np.mean(x)) / np.std(x)
print("Rows with |z| > 2 for distance_km:", (np.abs(z) > 2).sum())

Column means [age, distance_km, fee_inr, lead_days]: [ 48.17   9.5  635.71  11.82]
Rows with lead_days > 14: 296 (29.9%)
Rows with |z| > 2 for distance_km: 53


## Task 1.5 — Enrich the clinics from the Open-Meteo API
One retry per clinic; any failure stores `None` (never an invented value) and prints the reason.

In [11]:
clinics = pd.read_csv(CLINICS_FILE)
URL = "https://api.open-meteo.com/v1/forecast"

def fetch_temp(lat, lon):
    """Return (temp_c, reason). temp_c is None when the call fails."""
    params = {"latitude": lat, "longitude": lon,
              "hourly": "temperature_2m", "forecast_days": 1}
    reason = "unknown error"
    for attempt in (1, 2):                       # first try + one retry
        try:
            resp = requests.get(URL, params=params, timeout=10)
            print(f"   attempt {attempt}: status {resp.status_code}")
            if resp.status_code != 200:
                reason = f"HTTP status {resp.status_code}"
                continue
            data = resp.json()
            temps = data.get("hourly", {}).get("temperature_2m")
            if not temps:
                reason = "'hourly' data missing from reply"
                continue
            return temps[0], None
        except requests.exceptions.RequestException as e:
            reason = f"{type(e).__name__}: {e}"
    return None, reason

temps = []
for _, c in clinics.iterrows():
    print(c["clinic_name"])
    t, reason = fetch_temp(c["latitude"], c["longitude"])
    if t is None:
        print(f"   FAILED for {c['clinic_name']}: {reason} -> temp_c stored as None")
    temps.append(t)

clinics["temp_c"] = temps
clinics.to_csv("clinics_enriched.csv", index=False)
clinics

Kothrud Family Clinic
   attempt 1: status 200
Hadapsar Care Centre
   attempt 1: status 200
Sitabuldi Health Point
   attempt 1: status 200
Malviya Nagar Clinic
   attempt 1: status 200
Vaishali Nagar Clinic
   attempt 1: status 200
Edappally Medical Centre
   attempt 1: status 200
Vijay Nagar Clinic
   attempt 1: status 200
Gomti Nagar Care Point
   attempt 1: status 200


,clinic_id,clinic_name,city,tier,latitude,longitude,temp_c
0,C01,Kothrud Family Clinic,Pune,Metro,18.5074,73.8077,17.1
1,C02,Hadapsar Care Centre,Pune,Metro,18.5089,73.9260,18.3
2,C03,Sitabuldi Health Point,Nagpur,Tier-2,21.1458,79.0882,19.7
3,C04,Malviya Nagar Clinic,Jaipur,Tier-2,26.8553,75.8150,20.6
4,C05,Vaishali Nagar Clinic,Jaipur,Tier-2,26.9124,75.7873,20.6
5,C06,Edappally Medical Centre,Kochi,Tier-2,10.0261,76.3085,24.9
6,C07,Vijay Nagar Clinic,Indore,Tier-2,22.7533,75.8937,20.2
7,C08,Gomti Nagar Care Point,Lucknow,Tier-2,26.8467,80.9462,19.2


## Task 1.6 — Export the clean table and write the SQL load script

In [12]:
COLS = ["appointment_id", "patient_id", "clinic_id", "booked_on", "appt_date", "age", "gender",
        "distance_km", "fee_inr", "booking_channel", "visit_type", "reminder_sent",
        "prior_no_shows", "status", "lead_days", "no_show"]

clean = df.sort_values("appointment_id").copy()
clean["booked_on"] = clean["booked_on"].dt.strftime("%Y-%m-%d")
clean["appt_date"] = clean["appt_date"].dt.strftime("%Y-%m-%d")
clean = clean[COLS]

print("appointment_id unique:", clean["appointment_id"].is_unique,
      f"({clean['appointment_id'].nunique()} distinct)")
print("Missing values across the 16 columns:", int(clean.isnull().sum().sum()))
assert clean["appointment_id"].is_unique
assert clean.isnull().sum().sum() == 0

clean.to_csv("appointments_clean.csv", index=False)
print("Saved appointments_clean.csv with shape", clean.shape)

appointment_id unique: True (990 distinct)
Missing values across the 16 columns: 0
Saved appointments_clean.csv with shape (990, 16)


In [13]:
TEXT_APPT = {"appointment_id", "patient_id", "clinic_id", "booked_on", "appt_date",
             "gender", "booking_channel", "visit_type", "reminder_sent", "status"}
TEXT_CLINIC = {"clinic_id", "clinic_name", "city", "tier"}

def sql_value(col, value, text_cols):
    if pd.isna(value):
        return "NULL"                            # missing value, unquoted
    if col in text_cols:
        return f"'{value}'"
    return f"{value}"

lines = []
clinics_enriched = pd.read_csv("clinics_enriched.csv")
for _, r in clinics_enriched.iterrows():         # clinics first (foreign-key parent)
    vals = ",".join(sql_value(c, r[c], TEXT_CLINIC) for c in clinics_enriched.columns)
    lines.append(f"INSERT INTO clinics VALUES ({vals});")

for _, r in clean.iterrows():
    vals = ",".join(sql_value(c, r[c], TEXT_APPT) for c in COLS)
    lines.append(f"INSERT INTO appointments VALUES ({vals});")

with open("load_data.sql", "w") as f:
    f.write("\n".join(lines) + "\n")

n_clin = sum(l.startswith("INSERT INTO clinics") for l in lines)
n_appt = sum(l.startswith("INSERT INTO appointments") for l in lines)
print(f"load_data.sql: {n_clin} clinics + {n_appt} appointments = {len(lines)} INSERT statements")
print(lines[0])
print(lines[n_clin])

load_data.sql: 8 clinics + 990 appointments = 998 INSERT statements
INSERT INTO clinics VALUES ('C01','Kothrud Family Clinic','Pune','Metro',18.5074,73.8077,17.1);
INSERT INTO appointments VALUES ('A0001','P509','C05','2025-01-24','2025-01-25',60,'F',18.0,850,'Phone','Consultation','Yes',1,'No-Show',1,1);
